<pre>
- Poluição do ar - SO₂ 	(µg/m3)	-> Dióxido de Enxofre 
</pre>


In [ ]:
import cdsapi
import os, sys
import xarray as xr
from pyspark.sql import functions as F

In [ ]:
# Cria uma conexão SPARK

# Adiciona a pasta raiz do projeto (um ou dois níveis acima) no caminho do Python
sys.path.append(os.path.abspath(os.path.join('..')))  # Ajuste a quantidade de '..' conforme a profundidade da subpasta

# Cria uma conexão Spark 
from spark_utils import get_spark_session # ver em C:\Marco Conti\Projetos\MAIS-v2\spark_utils.py
spark = get_spark_session("Dioxido_enxofre")


In [ ]:
PROJECT_PATH   = os.getcwd()
DATA_PATH_ROOT = "C:\\Marco Conti\\Projetos\\Dados\\"

print(PROJECT_PATH)

In [ ]:
def get_EAC4_(year, client):
    dataset = "cams-global-reanalysis-eac4"
    request = {
        "variable": [
            "sulphur_dioxide"
        ],
        "pressure_level": ["1000"],
        "date": [f"{year}-12-01/{year}-12-31"],
        "time": ["06:00"],
        "data_format": "netcdf",
        "area": [6, -74, -35, -34]
    }

    ret_download = client.retrieve(dataset, request).download()
    return ret_download

def convert_nc_to_spark_dataframe(path, file_name):
    with xr.open_dataset(f"{path}\{file_name}"
                        ,engine="netcdf4"
                        ,chunks={"time": 365
                                ,"latitude": 100
                                ,"longitude": 100 }
                        ) as ds:
        # Transforma o Dataset em um Spark Dataframe
        df_dask            = ds.to_dask_dataframe()
        df_dask_c          = df_dask.compute()
        df_dioxido_enxofre = spark.createDataFrame(df_dask_c)

    return df_dioxido_enxofre

def convert_unit(df_dioxido_enxofre):
    # Densidade do ar em condições padrão (20°C, 1013.25 hPa) ~ 1.2041 kg/m3
    RHO_AIR_STD = 1.2041  # kg/m3
    FATOR_CONVERSAO_SO2 = RHO_AIR_STD * 1e9  # ~ 1.2041e9

    drop_cols = ["valid_time", "so2"]

    df_dioxido_enxofre_ug_m3 = \
        (df_dioxido_enxofre
            .withColumns({"data_medicao": F.col("valid_time").cast("date")
                        ,"indicador": F.lit("Poluição do ar - SO₂ (µg/m3)") 
                        ,"valor": (F.col("so2") * F.lit(FATOR_CONVERSAO_SO2)).cast("double")
                        ,"unidade_medida": F.lit("µg/m3")})
            .drop(*drop_cols)
        )

    return df_dioxido_enxofre_ug_m3

def write_data_csv(df_dioxido_enxofre_ug_m3, write_path, file_name):
    
    df_dioxido_enxofre_ug_m3.toPandas().to_csv(f"{write_path}\{file_name}")

In [ ]:
years_process = [2005, 2006, 2007, 2008, 2009
                ,2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019
                ,2020, 2021, 2022, 2023, 2024, 2025]


client = cdsapi.Client(
    url = "https://ads.atmosphere.copernicus.eu/api",
    key = "34161618-bf6b-41ca-9272-50b917f789b9"
)

for year in years_process:
    print(f"{year} - Convert unit mean and save NC files to CSV", end = "")

    # ret_download = get_EAC4_(year, client)

    df_dioxido_enxofre = \
        convert_nc_to_spark_dataframe(f"{DATA_PATH_ROOT}\EAC4-poluicao", f"EAC4_so2_{year}.nc")

    df_dioxido_enxofre_ug_m3 = convert_unit(df_dioxido_enxofre)

    csv_path      = r"{DATA_PATH_ROOT}\EAC4-poluicao\arquivos_csv\so2".format(DATA_PATH_ROOT = DATA_PATH_ROOT)
    csv_file_name = f"EAC4_so2_{year}.csv"

    write_data_csv(df_dioxido_enxofre_ug_m3, csv_path, csv_file_name)



    # os.rename(ret_download, f"{DATA_PATH_ROOT}\EAC4-poluicao\EAC4_so2_{year}.nc")

    print(f"{year} - completed", "\n")